# L10：CNN卷积神经网络


# L10：CNN卷积神经网络

## 学习目标

1. 理解卷积神经网络（CNN）相对于全连接网络的核心优势
2. 掌握卷积层、池化层的工作原理与计算方式
3. 理解填充（Padding）和步幅（Stride）对输出尺寸的影响
4. 掌握经典CNN架构（LeNet、AlexNet、VGG、ResNet）的设计思想
5. 能够使用PyTorch实现一个完整的CNN图像分类器

## 核心知识点

### 1. 为什么需要CNN？

全连接网络处理图像时的问题：
- 参数数量爆炸：一张 224x224x3 的图像展平后有 150,528 维，若隐藏层有 4096 个神经元，仅这一层就需要约 6 亿个参数
- 忽视空间结构：像素被拉成一维向量，丢失了相邻像素间的空间关系
- 缺乏平移不变性：无法自然地识别出现在不同位置的同一物体

CNN 通过三大核心思想解决了这些问题：
- **局部连接（Local Connectivity）**：每个神经元只连接输入的一个局部区域
- **参数共享（Parameter Sharing）**：同一卷积核在整个输入上共享权重
- **层级结构（Hierarchical Structure）**：逐层提取从低级到高级的特征

### 2. 卷积层详解

#### 二维卷积运算

给定输入特征图 I 和卷积核 K：

$$(I * K)_{i,j} = \sum_{m} \sum_{n} I_{i+m, j+n} \cdot K_{m,n}$$

#### 卷积的物理意义

卷积核相当于一个特征检测器。例如：
- 垂直边缘检测器：[[-1,0,1],[-2,0,2],[-1,0,1]]
- 水平边缘检测器：[[-1,-2,-1],[0,0,0],[1,2,1]]
- 高斯模糊核：[[1,2,1],[2,4,2],[1,2,1]]/16

#### 填充（Padding）

在输入周围补零，以控制输出尺寸：
- Valid（无填充）：输出尺寸 = floor((H - F + 1) / S)
- Same（输出与输入尺寸相同）：padding = (F - 1) / 2

其中 H 为输入高度，F 为卷积核尺寸，S 为步幅。

#### 步幅（Stride）

卷积核每次移动的像素数。步幅为2时，输出尺寸约为步幅为1时的一半。

#### 多通道卷积

若输入有 C 个通道，卷积核也有 C 个通道，输出为单通道。若有 K 个卷积核，则输出为 K 通道。

### 3. 池化层（Pooling Layer）

池化层用于降维和增强特征鲁棒性：

#### 最大池化（Max Pooling）

取池化窗口内的最大值，保留最显著的特征。


In [ ]:
输入:  [[1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16]]

池化窗口 2x2, 步幅 2:

输出:  [[6, 8],
        [14, 16]]


#### 平均池化（Average Pooling）

取池化窗口内的平均值，保留背景信息。


In [ ]:
输出:  [[3.5, 5.5],
        [11.5, 13.5]]


### 4. CNN架构演进

#### LeNet-5 (1998)

- 7层网络（不含输入层）
- 经典架构：conv1(6,5x5) -> pool1 -> conv2(16,5x5) -> pool2 -> fc(120) -> fc(84) -> fc(10)
- 用于MNIST手写数字识别

#### AlexNet (2012)

- 8层网络，ImageNet竞赛冠军（错误率16.4%，第二名26.2%）
- 关键创新：ReLU激活、Dropout、GPU并行训练、数据增强
- 参数约6000万

#### VGG-16 (2014)

- 16层网络，结构极其规整
- 全部使用 3x3 卷积核（两个 3x3 卷积堆叠等于一个 5x5 的感受野）
- 参数约1.38亿

#### ResNet (2015)

- 引入残差连接（Skip Connection）：H(x) = F(x) + x
- 解决深层网络退化问题（不是梯度消失，而是优化困难）
- 可训练超过1000层的网络

### 5. 卷积层参数计算

给定输入 H×W×C，卷积核 F×F×C×K：
- 参数量：F × F × C × K
- 输出尺寸：(H' × W' × K)，其中 H' = floor((H + 2P - F)/S + 1)

例如：输入 224×224×3，64个 7×7 卷积核，步幅2，填充3：
- 参数量：7×7×3×64 = 9,408
- 输出尺寸：floor((224 + 6 - 7)/2 + 1) = 112×112×64

## 代码示例

### 从零实现卷积层


In [ ]:
import numpy as np

def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    """
    将输入图像转换为列形式，方便高效地进行卷积运算
    这是在大多数深度学习框架中使用的实现方式

    参数：
        input_data: (N, C, H, W) 的4维数组
        filter_h: 滤波器高度
        filter_w: 滤波器宽度
        stride: 步幅
        pad: 填充大小

    返回：
        col: 转换后的2D数组
    """
    N, C, H, W = input_data.shape
    out_h = (H + 2 * pad - filter_h) // stride + 1
    out_w = (W + 2 * pad - filter_w) // stride + 1

    # 填充输入
    img = np.pad(input_data, [(0, 0), (0, 0), (pad, pad), (pad, pad)], mode='constant')
    # 将每个滑动窗口展开为一行
    cols = np.zeros((N, out_h, out_w, filter_h, filter_w, C))

    for y in range(out_h):
        for x in range(out_w):
            y_start = y * stride
            x_start = x * stride
            cols[:, y, x] = img[:, :, y_start:y_start+filter_h, x_start:x_start+filter_w]

    cols = cols.transpose(0, 4, 1, 2, 5, 3).reshape(N * out_h * out_w, -1)
    return cols, out_h, out_w


def col2im(col, input_shape, filter_h, filter_w, stride=1, pad=0):
    """
    将列形式的梯度转回原始图像形式
    """
    N, C, H, W = input_shape
    out_h = (H + 2 * pad - filter_h) // stride + 1
    out_w = (W + 2 * pad - filter_w) // stride + 1
    col = col.reshape(N, out_h, out_w, filter_h, filter_w, C)
    col = col.transpose(0, 5, 1, 2, 3, 4)

    img = np.zeros((N, C, H + 2 * pad + stride - 1, W + 2 * pad + stride - 1))
    for y in range(out_h):
        for x in range(out_w):
            y_start = y * stride
            x_start = x * stride
            img[:, :, y_start:y_start+filter_h, x_start:x_start+filter_w] += col[:, :, y, x]

    return img[:, :, pad:H+pad, pad:W+pad]


class Conv2D:
    """2D卷积层实现"""

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # Xavier初始化卷积核
        fan_in = in_channels * kernel_size * kernel_size
        scale = np.sqrt(2.0 / fan_in)
        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * scale
        self.b = np.zeros((out_channels, 1))

        # 用于反向传播的缓存
        self.x_cache = None
        self.col_cache = None

    def forward(self, x):
        """
        前向传播
        参数：
            x: (N, C, H, W) 的输入
        返回：
            out: (N, K, H', W') 的输出
        """
        N, C, H, W = x.shape

        # im2col转换
        self.x_cache = x
        col, out_h, out_w = im2col(x, self.kernel_size, self.kernel_size,
                                    self.stride, self.padding)
        self.col_cache = col

        # 卷积核也转为列形式：(out_channels, in_channels*kernel_h*kernel_w)
        W_col = self.W.reshape(self.out_channels, -1)

        # 计算输出
        out = col @ W_col.T + self.b.reshape(-1)
        out = out.reshape(N, out_h, out_w, self.out_channels)
        out = out.transpose(0, 3, 1, 2)  # 转为 (N, K, H', W')

        return out

    def backward(self, d_out):
        """
        反向传播
        参数：
            d_out: (N, K, H', W') 的梯度
        返回：
            d_x: (N, C, H, W) 的梯度
        """
        N, K, _, _ = d_out.shape

        # 梯度转为列形式
        d_out_col = d_out.transpose(0, 2, 3, 1).reshape(-1, self.out_channels)

        # 梯度计算
        dW = d_out_col.T @ self.col_cache  # (K, C*F*F)
        dW = dW.reshape(self.out_channels, self.in_channels, self.kernel_size, self.kernel_size)
        db = np.sum(d_out_col, axis=0).reshape(-1, 1)

        # 输入梯度（通过col2im转回图像形式）
        W_col = self.W.reshape(self.out_channels, -1)
        d_col = d_out_col @ W_col  # (N*H'*W', C*F*F)
        d_x = col2im(d_col, self.x_cache.shape, self.kernel_size, self.kernel_size,
                     self.stride, self.padding)

        self.grad_W = dW / N
        self.grad_b = db / N

        return d_x

    def update(self, lr=0.01):
        """参数更新"""
        self.W -= lr * self.grad_W
        self.b -= lr * self.grad_b


class MaxPool2D:
    """最大池化层实现"""

    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride = stride
        self.x_cache = None
        self.argmax_cache = None

    def forward(self, x):
        """前向传播"""
        N, C, H, W = x.shape
        out_h = H // self.stride
        out_w = W // self.stride

        # 重塑为池化窗口
        out = np.zeros((N, C, out_h, out_w))
        self.argmax_cache = np.zeros((N, C, out_h, out_w), dtype=int)

        for y in range(out_h):
            for x_idx in range(out_w):
                y_start = y * self.stride
                x_start = x_idx * self.stride
                window = x[:, :, y_start:y_start+self.pool_size, x_start:x_start+self.pool_size]
                # 记录最大值的位置
                self.argmax_cache[:, :, y, x_idx] = np.argmax(
                    window.reshape(N, C, -1), axis=2)
                out[:, :, y, x_idx] = np.max(window, axis=(2, 3))

        self.x_cache = x
        return out

    def backward(self, d_out):
        """反向传播"""
        N, C, out_h, out_w = d_out.shape
        d_x = np.zeros_like(self.x_cache)

        for y in range(out_h):
            for x_idx in range(out_w):
                y_start = y * self.stride
                x_start = x_idx * self.stride
                window_size = self.pool_size * self.pool_size

                # 重建梯度矩阵
                grad_window = np.zeros((N, C, window_size))
                argmax = self.argmax_cache[:, :, y, x_idx].flatten()
                grad_window[np.arange(len(argmax)), argmax] = d_out[:, :, y, x_idx].flatten()

                grad_window = grad_window.reshape(N, C, self.pool_size, self.pool_size)
                d_x[:, :, y_start:y_start+self.pool_size, x_start:x_start+self.pool_size] = grad_window

        return d_x


class BatchNorm2D:
    """2D数据的批归一化层"""

    def __init__(self, num_channels, momentum=0.9, epsilon=1e-5):
        self.num_channels = num_channels
        self.momentum = momentum
        self.epsilon = epsilon

        # 可学习参数
        self.gamma = np.ones((1, num_channels, 1, 1))
        self.beta = np.zeros((1, num_channels, 1, 1))

        # 运行统计量（用于推理）
        self.running_mean = np.zeros((1, num_channels, 1, 1))
        self.running_var = np.ones((1, num_channels, 1, 1))

    def forward(self, x, training=True):
        """前向传播"""
        if training:
            # 计算批统计量
            mu = np.mean(x, axis=(0, 2, 3), keepdims=True)
            var = np.var(x, axis=(0, 2, 3), keepdims=True)

            # 更新运行统计量
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * mu
            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * var

            self.cache = (x, mu, var)
        else:
            mu = self.running_mean
            var = self.running_var

        # 标准化
        x_norm = (x - mu) / np.sqrt(var + self.epsilon)
        return self.gamma * x_norm + self.beta

    def backward(self, d_out):
        """反向传播"""
        x, mu, var = self.cache
        m = x.shape[0] * x.shape[2] * x.shape[3]

        # 简化版本（实际实现更复杂）
        dx_norm = d_out * self.gamma
        dvar = np.sum(dx_norm * (x - mu) * (-0.5) * (var + self.epsilon) ** (-3/2), axis=(0, 2, 3), keepdims=True)
        dmu = np.sum(dx_norm * (-1) / np.sqrt(var + self.epsilon), axis=(0, 2, 3), keepdims=True) + \
              dvar * np.mean(-2 * (x - mu), axis=(0, 2, 3), keepdims=True)
        dx = dx_norm / np.sqrt(var + self.epsilon) + dvar * 2 * (x - mu) / m + dmu / m

        return dx


### 使用PyTorch构建CNN


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# ============================================================
# CNN模型定义
# ============================================================

class CNN(nn.Module):
    """
    经典CNN架构：卷积层 + 批归一化 + ReLU + 池化
    结构参考VGG，但规模缩小以适应MNIST
    """

    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        # 特征提取部分（VGG风格：多个3x3卷积堆叠）
        self.features = nn.Sequential(
            # Block 1: 1 -> 64 channels
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 28x28 -> 14x14

            # Block 2: 64 -> 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 14x14 -> 7x7

            # Block 3: 128 -> 256 channels
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 7x7 -> 3x3
        )

        # 分类器部分
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 3 * 3, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

        # 权重初始化
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x


class ResidualBlock(nn.Module):
    """ResNet的残差块"""

    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Shortcut connection
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += self.shortcut(identity)
        out = self.relu(out)

        return out


class SimpleResNet(nn.Module):
    """简化的ResNet用于CIFAR-10"""

    def __init__(self, num_classes=10):
        super(SimpleResNet, self).__init__()

        self.in_channels = 64

        # 初始卷积层
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        # 残差层
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)

        # 全局平均池化
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)

    def _make_layer(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []

        for s in strides:
            layers.append(ResidualBlock(self.in_channels, out_channels, s))
            self.in_channels = out_channels

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x


# ============================================================
# 训练函数
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    """训练一个epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()

    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader, criterion, device):
    """评估模型"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

    return total_loss / len(loader), 100. * correct / total


# ============================================================
# 主程序
# ============================================================

if __name__ == "__main__":
    # 检测设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")

    # 加载数据（示例使用torchvision实际加载）
    # from torchvision import datasets, transforms
    # transform = transforms.Compose([
    #     transforms.RandomCrop(32, padding=4),
    #     transforms.RandomHorizontalFlip(),
    #     transforms.ToTensor(),
    #     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    # ])
    # train_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform)
    # test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform)
    # train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4)
    # test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4)

    # 演示：使用随机数据模拟训练流程
    print("\n创建模型...")
    model = CNN(num_classes=10).to(device)
    print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

    # 定义损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    # 模拟训练循环
    print("\n开始训练（模拟）...")
    for epoch in range(20):
        train_loss, train_acc = train_epoch(model, None, criterion, optimizer, device)
        scheduler.step()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Acc={train_acc:.2f}%")

    # 展示模型结构
    print("\n模型结构:")
    print(model)


## 练习题

1. **卷积计算**：假设输入图像尺寸为 32×32×3，使用 64 个 5×5×3 的卷积核，步幅为1，填充为2。请问输出特征图的尺寸是多少？该卷积层有多少参数？

2. **感受野分析**：一个3×3卷积核的堆叠（conv1 -> conv2 -> conv3，步幅均为1，无填充）能够覆盖多大的输入区域？这对于图像分类意味着什么？

3. **残差连接**：解释为什么残差连接能够缓解深层网络的训练困难。如果跳跃连接维度不匹配，有哪些处理方式？

4. **架构设计**：如果要构建一个网络来识别不同种类的花（5个类别），图像尺寸为224×224×3。请设计一个合适的CNN架构，并说明为什么这样设计。

5. **参数量对比**：分别计算以下两种架构的参数量：(1) 输入-全连接层(4096)-输出层(1000)；(2) 输入-卷积层(3×3×64)-池化-卷积层(3×3×64)-池化-全连接层(1000)。假设输入图像为224×224×3。

## 延伸阅读

- LeCun, Y., et al. (1989). "Backpropagation Applied to Handwritten Zip Code Recognition".
- Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). "ImageNet Classification with Deep Convolutional Neural Networks" (AlexNet). NIPS 2012.
- Simonyan, K., & Zisserman, A. (2014). "Very Deep Convolutional Networks for Large-Scale Image Recognition" (VGG). ICLR 2015.
- He, K., et al. (2016). "Deep Residual Learning for Image Recognition" (ResNet). CVPR 2016.
- 经典CNN可视化：Zeiler, M. D., & Fergus, R. (2014). "Visualizing and Understanding Convolutional Networks". ECCV 2014.

---
